In [ ]:
!python settings.py

In [ ]:
import os
import json
import pandas as pd
from pprint import pprint
from tqdm.autonotebook import tqdm

from sentence_transformers import SentenceTransformer
from mteb import MTEB
from mteb.abstasks.TaskMetadata import TaskMetadata
from mteb.abstasks.AbsTaskRetrieval import AbsTaskRetrieval

from settings import MODEL_NAME, OUTPUT_DIR, DEVICE, BATCH_SIZE

os.environ['WANDB_DISABLED'] = 'true'

In [ ]:
data = {
    'corpus': pd.read_parquet('data/processed/corpus_data.parquet'),
    'train' : pd.read_parquet('data/processed/train_data.parquet'),
    'test'  : pd.read_parquet('data/processed/test_data.parquet')
}
for split in ['train', 'test']:
    data[split]['cid']          = data[split]['cid'].apply(lambda x: x.tolist())
    data[split]['context_list'] = data[split]['context_list'].apply(lambda x: x.tolist())

In [ ]:
class BKAILegalDocRetrievalTask(AbsTaskRetrieval):
    # Metadata definition used by MTEB benchmark
    metadata = TaskMetadata(name='BKAILegalDocRetrieval',
                            description='',
                            reference='https://github.com/embeddings-benchmark/mteb/blob/main/docs/adding_a_dataset.md',
                            type='Retrieval',
                            category='s2p',
                            modalities=['text'],
                            eval_splits=['test'],
                            eval_langs=['vi'],
                            main_score='ndcg_at_10',
                            other_scores=['recall_at_10', 'precision_at_10', 'map'],
                            dataset={
                                'path'    : 'data',
                                'revision': 'd4c5a8ba10ae71224752c727094ac4c46947fa29',
                            },
                            date=('2012-01-01', '2020-01-01'),
                            form='Written',
                            domains=['Academic', 'Non-fiction'],
                            task_subtypes=['Scientific Reranking'],
                            license='cc-by-nc-4.0',
                            annotations_creators='derived',
                            dialect=[],
                            text_creation='found',
                            bibtex_citation=''
    )

    data_loaded = True # Flag

    def __init__(self, **kwargs):
        super().__init__(**kwargs)

        self.corpus        = {}
        self.queries       = {}
        self.relevant_docs = {}

        shared_corpus = {}
        for _, row in data['corpus'].iterrows():
            shared_corpus[f"c{row['cid']}"] = {
                'text': row['text'],
                '_id' : row['cid']
            }
            
        for split in ['train', 'test']:
            self.corpus[split]        = shared_corpus
            self.queries[split]       = {}
            self.relevant_docs[split] = {}

        for split in ['train', 'test']:
            for _, row in data[split].iterrows():
                qid, cids = row['qid'], row['cid']
                
                qid_str   = f'q{qid}'
                cids_str  = [f'c{cid}' for cid in cids]
                
                self.queries[split][qid_str] = row['question']
                
                if qid_str not in self.relevant_docs[split]:
                    self.relevant_docs[split][qid_str] = {}
                    
                for cid_str in cids_str:
                    self.relevant_docs[split][qid_str][cid_str] = 1
            
        self.data_loaded = True

In [ ]:
fine_tuned_model = SentenceTransformer(OUTPUT_DIR, device=DEVICE)

In [ ]:
custom_task = BKAILegalDocRetrievalTask()
evaluation  = MTEB(tasks=[custom_task])
evaluation.run(fine_tuned_model, batch_size=BATCH_SIZE)

In [ ]:
file_path = f"results/{MODEL_NAME}/no_revision_available/BKAILegalDocRetrieval.json"

with open(file_path, 'r', encoding='utf-8') as f:
    eval_data = json.load(f)

scores = eval_data["scores"]["test"][0]
main_metrics = {
    'main_score'         : scores.get('ndcg_at_10'),
    'recall@10'          : scores.get('recall_at_10'),
    'precision@10'       : scores.get('precision_at_10'),
    'mrr@10'             : scores.get('mrr_at_10'),
    'evaluation_time (s)': eval_data.get('evaluation_time')
}

print('Main Evaluation Metrics (Top-K = 10):')
pprint(main_metrics)

In [ ]:
metrics = {k: v for k, v in scores.items() if '_at_' in k and not k.startswith('nauc')}

parsed_metrics = []
for key, value in metrics.items():
    metric, at_k = key.split('_at_')
    parsed_metrics.append({'metric': metric, 'k': int(at_k), 'score': value})

df_metrics = pd.DataFrame(parsed_metrics).pivot(index='k', columns='metric', values='score')
df_metrics = df_metrics.sort_index()

print("\nEvaluation Scores by K:")
print(df_metrics.round(4))